Anzal Azam Shaikh

 DHC - 1338

 Link to access: https://colab.research.google.com/drive/188T3E2ht1mGbr3JKsOgY4hN1qRLli6i9?usp=sharing

Objective:
Build a reusable and production-ready machine learning pipeline for predicting customer
churn.

Dataset:
Telco Churn Dataset

Instructions:

● Implement data preprocessing steps (e.g., scaling, encoding) using Pipeline

● Train models like Logistic Regression and Random Forest

● Use GridSearchCV for hyperparameter tuning

● Export the complete pipeline using joblib

In [1]:
# 1 — Install Libraries
!pip install -q scikit-learn pandas joblib

In [2]:
# 2 — Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
import joblib

In [7]:
import os

print(os.listdir())

['.config', 'archive (4).zip', 'sample_data']


In [9]:
# Unzip file
import zipfile

with zipfile.ZipFile("archive (4).zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [10]:
# Check extracted files
import os

print(os.listdir("data"))

['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [11]:
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
# 4 — Data Cleaning
# Remove customer ID (not useful)
df.drop("customerID", axis=1, inplace=True)

# Convert target column
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [13]:
# 5 — Split Features
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [14]:
# 6 — Identify Column Types
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

In [15]:
# 7 — Preprocessing Pipeline
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

In [16]:
# 8 — Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
# 9 — Model Pipeline (Logistic Regression)
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [18]:
# 10 — Random Forest Pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier())
])

In [19]:
# 11 — Train Models
log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier())])

In [20]:
# 12 — Evaluate Models
log_pred = log_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Logistic Regression Accuracy: 0.8239886444286728
Random Forest Accuracy: 0.7984386089425124


In [21]:
# 13 — Hyperparameter Tuning (GridSearchCV)
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                                       ('classifier',
                                        RandomForestClassifier())]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__n_estimators': [100, 200]},
             scoring='accuracy')

In [22]:
# 14 — Best Model
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Params: {'classifier__max_depth': None, 'classifier__n_estimators': 200}
Best Score: 0.7919772807951722


In [23]:
# 15 — Save Final Pipeline
joblib.dump(grid.best_estimator_, "churn_pipeline.pkl")

['churn_pipeline.pkl']

In [26]:
# 16 — Load & Predict (REUSABILITY TEST)
model = joblib.load("churn_pipeline.pkl")

sample = X_test.iloc[0:10]
prediction = model.predict(sample)

print("Prediction:", prediction)

Prediction: [1 0 0 1 0 0 0 0 0 0]


FINAL OUTPUT

Preprocessing using Pipeline

Logistic Regression model

Random Forest model

GridSearchCV tuning

Export using joblib

Reusable production pipeline